# Exercise 2.2: Data Types and Subsetting (Angola IEA)

This notebook continues with the Q4 2025 IEA file. Using only the tools from
Lesson 2.2, you fix the data types and save a **typed checkpoint** that the
cleaning step (2.3) picks up.

You will practice:
- Telling a DataFrame from a Series, and checking dtypes
- Renaming 29 Portuguese variable names to readable English ones
- Keeping identifiers as text, including the float to int to string route
- Converting a YYYYMMDD number into a real date with `pd.to_datetime()`
- Saving memory with the `category` dtype
- Subsetting to inspect problems, not yet to fix them
- Saving a typed checkpoint to `10_cleaned/`

> **Pipeline:** reads `0_raw/`, writes `10_cleaned/angola_iea_2025q4_typed.csv`.
> Exercise 2.3 reads that file.

### Path Setup (run first)

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola/employment_survey'
DATA_CLEAN_DIR = '../../data/10_cleaned'
RAW_FILE = 'IEA_2025_IV_TRIM_IND.sav'
raw_path = os.path.join(DATA_RAW_DIR, RAW_FILE)

SPSS_COLS = [
    'NIDF', 'PPNO', 'G_06_ID_IEA', 'PROV', 'AREA_RESID', 'G_15_TRIMESTRE',
    'DEM_REL', 'DEM_SEX', 'DEM_AGE', 'DEM_MRT', 'DEM_EDL', 'S03_01',
    'ATW_PAY', 'ATW_PFT', 'ATW_FAM', 'ABS_JOB',
    'SRH_JOB', 'SRH_BUS', 'SRH_AVN', 'SRH_AVL', 'SRH_DES',
    'WKT_USHRSTOT', 'WKT_ACHRSTOT', 'MJT_SYR', 'MJJ_EMP_REL', 'GHVEDT',
    'POND_IEA_IV_TRIM_2025_IND', 'G_12', 'G_13',
]

df = pd.read_spss(raw_path, usecols=SPSS_COLS, convert_categoricals=False)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded:', df.shape)
df.head()

---

## Task 1: DataFrame vs Series

Selecting one column returns a **Series**, a one dimensional object with its own
dtype and its own methods. `.str`, `.dt` and `.value_counts()` all belong to
Series, not to the whole DataFrame.

In [ ]:
print(type(df))
print(type(df['DEM_AGE']))

In [ ]:
df.  # your code here

**Questions:**

- Every one of the 29 columns comes back as the same dtype. Which one, and why?
- Why is `float64` the wrong dtype for `NIDF`, the household identifier?
- `GHVEDT` is a date stored as the number 20251204. What has to happen before
  anything date-like will work on it?

---

## Task 2: Rename the columns

The SPSS names come from the questionnaire, not from the analysis. `PROV` and
`ATW_PAY` are precise but unreadable. Rename once, here, and every later notebook
is easier to follow.

In [ ]:
RENAME_MAP = {
    'NIDF': 'household_id', 'PPNO': 'person_no', 'G_06_ID_IEA': 'cluster_id',
    'PROV': 'province_code', 'AREA_RESID': 'area_type', 'G_15_TRIMESTRE': 'quarter',
    'DEM_REL': 'rel_to_head', 'DEM_SEX': 'sex', 'DEM_AGE': 'age',
    'DEM_MRT': 'marital_status', 'DEM_EDL': 'education_level',
    'S03_01': 'school_attendance', 'ATW_PAY': 'worked_for_pay',
    'ATW_PFT': 'worked_own_account', 'ATW_FAM': 'worked_family_business',
    'ABS_JOB': 'absent_from_job', 'SRH_JOB': 'sought_work',
    'SRH_BUS': 'sought_business', 'SRH_AVN': 'available_now',
    'SRH_AVL': 'available_2wk', 'SRH_DES': 'wants_work',
    'WKT_USHRSTOT': 'hours_usual', 'WKT_ACHRSTOT': 'hours_actual',
    'MJT_SYR': 'job_start_year', 'MJJ_EMP_REL': 'employment_relation',
    'GHVEDT': 'interview_date', 'POND_IEA_IV_TRIM_2025_IND': 'weight_ind',
    'G_12': 'hh_size_reported', 'G_13': 'hh_adults_reported',
}

df = df.  # your code here: rename with RENAME_MAP
print(df.columns.tolist())

**Questions:**

- After renaming, how many of the 29 columns changed name? What happens if a key
  in `RENAME_MAP` has a typo?
- Why are `available_now` and `available_2wk` kept as two separate columns
  instead of being merged into one?

---

## Task 3: Keep identifiers as text

`household_id` arrives as `9250068.0`. Converting straight to string would keep
the `.0`, so the route is float to integer to string.

`province_code` gets the same treatment plus `zfill(2)`. Angola's province codes
run 10 to 30, so `zfill(2)` changes nothing today. It is a documented safeguard,
the same way you would pad any code that could gain a single digit value later.

In [ ]:
print('Before:', df['household_id'].head(3).tolist())

for col in ['household_id', 'person_no', 'cluster_id']:
    df[col] = df[col].  # your code here: int64 then string

df['province_code'] =   # your code here: int64, string, then .str.zfill(2)

print('After: ', df['household_id'].head(3).tolist())
print('Provinces:', sorted(df['province_code'].unique()))

**Questions:**

- What string do you get if you skip the `int64` step and convert the float
  straight to `string`? Why is that a problem for joining?
- What range do the province codes cover, and how many distinct provinces are
  there? Is `zfill(2)` doing anything on this file today?
- Why should an identifier never be stored as a number?

---

## Task 4: Convert the interview date

`interview_date` is the float `20251204.0`, meaning 2025-12-04. Convert through
`Int64` (which tolerates the missing values) to string, then parse with an
explicit format.

In [ ]:
print('Raw values:', df['interview_date'].dropna().head(3).tolist())

date_text = df['interview_date'].astype('Int64').astype('string')
df['interview_date'] = pd.to_datetime(  # your code here: format='%Y%m%d', errors='raise' )

print('dtype:', df['interview_date'].dtype)
print('Range:', df['interview_date'].min(), 'to', df['interview_date'].max())
print('Missing (NaT):', df['interview_date'].isna().sum())

In [ ]:
# The .dt accessor unlocks date parts
print(df['interview_date'].dt.month.value_counts(dropna=False).sort_index())

**Questions:**

- What is the min and max of `interview_date` after conversion? Is that range
  plausible for a survey labelled 4th quarter 2025?
- How many rows have a missing (`NaT`) interview date, and why would the date be
  missing for some people but not others?
- What is `NaT`, and how does it behave in arithmetic compared to `NaN`?

---

## Task 5: Save memory with `category`

`area_type` holds two distinct values repeated 53,353 times. The `category` dtype
stores each label once and keeps small integer codes alongside.

In [ ]:
before = df['area_type'].memory_usage(deep=True)
after = df['area_type'].  # your code here: astype('category').memory_usage(deep=True)

print(f'float64:  {before:,} bytes')
print(f'category: {after:,} bytes')
print(f'Saved:    {(1 - after / before) * 100:.1f}%')

**Questions:**

- How many bytes does `area_type` use as `float64` versus as `category`? What is
  the percentage saved?
- Why is the `category` conversion not carried into the saved checkpoint file?

---

## Task 6: Subset to inspect the problems

Filtering here is for **looking**, not fixing. 2.3 makes the removal decisions.

In [ ]:
# Implausible working weeks
print('hours_usual > 100:', (df['hours_usual'] > 100).sum())
df[df['hours_usual'] > 100][['household_id', 'hours_usual', 'hours_actual']].head()

In [ ]:
# Combine conditions: each one needs its own parentheses
old_and_working = df[(df['age'] >= 65) & (df['hours_usual'] > 40)]
print('People 65+ working over 40 hours:', len(old_and_working))
old_and_working[['household_id', 'age', 'hours_usual']].head()

In [ ]:
# isin() for a set of provinces: Luanda and Benguela
target = df[df['province_code'].  # your code here: isin(['14', '23'])]
print('Rows in Luanda or Benguela:', len(target))

# str.contains() is safe with missing values when na=False
print('Codes containing "1":', df[df['province_code'].str.contains('1', na=False)]['province_code'].nunique())

**Questions:**

- How many people report more than 100 usual hours a week?
- How many rows fall in Luanda or Benguela combined?
- Why does each condition need its own parentheses when combined with `&`?
- What does `na=False` do inside `str.contains()`, and why is it needed here?

---

## Task 7: Save the typed checkpoint

Types are fixed. Save so 2.3 starts from a stable baseline.

> Never write into `0_raw/`. This goes to `10_cleaned/`.

In [ ]:
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)
out_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_typed.csv')

df.  # your code here: to_csv with index=False
print('Saved:', out_path, '|', df.shape)

In [ ]:
check = pd.read_csv(out_path, dtype={
    'household_id': 'string', 'person_no': 'string',
    'cluster_id': 'string', 'province_code': 'string',
})
print('Reloaded:', check.shape)
print('interview_date dtype after reload:', check['interview_date'].dtype)
check[['household_id', 'province_code', 'interview_date']].head()

**Questions:**

- What is the shape of the saved checkpoint? Has any row been removed at this
  point?
- After reloading the CSV, what dtype does `interview_date` come back as? What
  does that tell you about what CSV can and cannot store?
- What does `index=False` prevent when saving?